In [1]:
from notebookutils import mssparkutils
from datetime import datetime

try:
    batch_id = mssparkutils.widgets.get("batch_id")
except Exception:
    batch_id = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
print("batch_id = ", batch_id)

StatementMeta(, 10eb86a1-eebf-4950-a962-31894164c44e, 3, Finished, Available, Finished, False)

batch_id =  20260312_173009


In [8]:
from  pyspark.sql import functions as F

raw_checkin_path = "Files/yelp/raw/yelp_checkin.json"

df_checkin_raw = spark.read.json(raw_checkin_path)
df_checkin_raw.printSchema()
print("raw checkin count: ", df_checkin_raw.count())



StatementMeta(, 10eb86a1-eebf-4950-a962-31894164c44e, 10, Finished, Available, Finished, False)

root
 |-- business_id: string (nullable = true)
 |-- date: string (nullable = true)

raw checkin count:  131930


In [9]:
df_checkin_bronze = (
    df_checkin_raw
    .withColumn("_ingest_ts",F.current_timestamp())
    .withColumn("_source_file", F.input_file_name())
    .withColumn("_batch_id", F.lit(batch_id))
)


StatementMeta(, 10eb86a1-eebf-4950-a962-31894164c44e, 11, Finished, Available, Finished, False)

In [12]:
(df_checkin_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("checkin_bronze")
)

StatementMeta(, 10eb86a1-eebf-4950-a962-31894164c44e, 14, Finished, Available, Finished, False)

In [13]:
df_checkin_bronze.select("business_id", "date","_ingest_ts","_source_file","_batch_id")

StatementMeta(, 10eb86a1-eebf-4950-a962-31894164c44e, 15, Finished, Available, Finished, False)

DataFrame[business_id: string, date: string, _ingest_ts: timestamp, _source_file: string, _batch_id: string]

In [14]:
spark.table("checkin_bronze").printSchema()

StatementMeta(, 10eb86a1-eebf-4950-a962-31894164c44e, 16, Finished, Available, Finished, False)

root
 |-- business_id: string (nullable = true)
 |-- date: string (nullable = true)
 |-- _ingest_ts: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _batch_id: string (nullable = true)



In [ ]:
#Return to 01_0_bronze_run_all
mssparkutils.notebook.exit("SUCCESS")